<a href="https://colab.research.google.com/github/Doan-Truc/Doan-Truc/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import json
from collections import Counter
from scipy.signal import butter, filtfilt, welch
from sklearn.decomposition import FastICA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report
import os


In [ ]:
fs = 128  # Sampling rate
run_names = ['Run1', 'Run2', 'Run3', 'Run4', 'Run5','Run6','Run7','Run8']

# Lấy đúng các nhãn hành động MI
valid_labels = ['3', '4', '5', '6']
label_map = {
    '3': 'Right Hand',
    '4': 'Left Hand',
    '5': 'Right Foot',
    '6': 'Left Foot'
}


Bandpass, ICA, Chuẩn hoá

In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA

# ==== CONFIG ====
fs = 128  # Hz
run_names = ['Run1', 'Run2', 'Run3', 'Run4', 'Run5']
base_path = "/content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024"
output_dir = os.path.join(base_path, "cleaned_data")
os.makedirs(output_dir, exist_ok=True)

# === Labels of interest ===
valid_labels = ['3', '4', '5', '6']
label_map = {
    '3': 'Right Hand',
    '4': 'Left Hand',
    '5': 'Right Foot',
    '6': 'Left Foot'
}

# ==== FILTER FUNCTIONS ====
def bandpass_filter(eeg, low=8, high=30, fs=128, order=4):
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype='band')
    return filtfilt(b, a, eeg, axis=0)

def apply_ica(eeg, n_components=None):
    ica = FastICA(n_components=n_components or eeg.shape[1], random_state=42)
    return ica.fit_transform(eeg)

def normalize(eeg):
    scaler = StandardScaler()
    return scaler.fit_transform(eeg)

# ==== PIPELINE ====
for run in run_names:
    print(f"\n🧠 Processing {run} ...")

    # === Load files ===
    csv_file = os.path.join(base_path, f"{run}_Raw Signals.csv")
    label_file = os.path.join(base_path, f"{run}_Action label.txt")
    event_file = os.path.join(base_path, f"{run}_Event timestamp.txt")
    json_file = os.path.join(base_path, f"{run}_Session setup.json")

    if not all(os.path.exists(f) for f in [csv_file, label_file, event_file, json_file]):
        print(f"⚠️ Missing file(s) for {run}, skipping...")
        continue

    eeg = pd.read_csv(csv_file, header=None).iloc[:, :22].values
    labels = [line.strip() for line in open(label_file)]
    events = [int(line.strip()) for line in open(event_file)]
    session_info = json.load(open(json_file))

    # === Step 1: Bandpass filter ===
    eeg_filtered = bandpass_filter(eeg, low=8, high=30, fs=fs)

    # === Step 2: ICA (artifact removal) ===
    eeg_ica = apply_ica(eeg_filtered)

    # === Step 3: Normalization ===
    eeg_norm = normalize(eeg_ica)

    # === Step 4: Segment trials for MI actions ===
    trial_signals = []
    trial_labels = []

    trial_len = 12 * fs  # 12 seconds per trial
    for i, start in enumerate(events):
        label = labels[i * 4 + 2] if (i * 4 + 2) < len(labels) else None  # The 3rd action per trial (RH/LH/RF/LF)
        if label in valid_labels:
            seg = eeg_norm[start:start + trial_len, :]
            if seg.shape[0] == trial_len:
                trial_signals.append(seg)
                trial_labels.append(label_map[label])

    trial_signals = np.array(trial_signals)
    trial_labels = np.array(trial_labels)

    # === Step 5: Save cleaned EEG ===
    save_path = os.path.join(output_dir, f"{run}_cleaned.npy")
    np.save(save_path, {"signals": trial_signals, "labels": trial_labels})
    print(f"✅ Saved cleaned EEG: {save_path}")
    print(f"   -> Trials: {len(trial_labels)} valid ({trial_labels})")



🧠 Processing Run1 ...
✅ Saved cleaned EEG: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_data/Run1_cleaned.npy
   -> Trials: 3 valid (['Right Hand' 'Right Hand' 'Right Hand'])

🧠 Processing Run2 ...
✅ Saved cleaned EEG: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_data/Run2_cleaned.npy
   -> Trials: 3 valid (['Right Hand' 'Right Hand' 'Right Hand'])

🧠 Processing Run3 ...
✅ Saved cleaned EEG: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_data/Run3_cleaned.npy
   -> Trials: 3 valid (['Right Hand' 'Right Hand' 'Right Hand'])

🧠 Processing Run4 ...
✅ Saved cleaned EEG: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_data/Run4_cleaned.npy
   -> Trials: 3 valid (['Right Hand' 'Right Hand' 'Right Hand'])

🧠 Processing Run5 ...
✅ Saved cleaned EEG: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_data/Run5_cleaned.npy
   -> Trials: 3 valid (['Right Hand' 'Right Hand' 'Right Hand'])
